# GB1 ESM-2 Quickstart: Best Fused CNN Model

This notebook is intended to be the **small, grader-friendly reproduction notebook** for the GB1 multi-mutation prediction project.

It does three things:

1. Loads the original GB1 split/fitness metadata and/or the saved ESM-2 embeddings.
2. Reconstructs the final feature representation used for the fused models: full-sequence mean embedding plus an unflattened local window around the four GB1 mutation sites.
3. Trains the best fused CNN local model across the official FLIP splits: `one_vs_rest`, `two_vs_rest`, and `three_vs_rest`.

The model reproduced here is the best CNN configuration from the final architecture sweep:

- CNN channels: `384`
- CNN kernel size: `7`
- CNN layers: `1`
- Global branch: `(128, 64)`
- Fusion branch: `(64,)`
- Dropout: `0.1`
- Optimizer: AdamW, `lr=1e-3`, `weight_decay=1e-3`

Note: the **feature window radius** is `WINDOW_RADIUS = 5`, while the **CNN kernel size** is `CNN_KERNEL_SIZE = 7`. These are different parameters.


## 1. Install packages

In Colab, PyTorch is usually already installed. If the PyTorch install line causes runtime or CUDA compatibility issues, comment it out and restart the runtime.


In [ ]:
# Core scientific stack
!pip -q install numpy pandas scipy scikit-learn matplotlib tqdm

# PyTorch is usually pre-installed in Colab. Uncomment if needed.
# !pip -q install torch torchvision torchaudio


## 2. Imports, paths, and configuration

Expected repository layout:

```text
repo/
  data/
    gb1_esm2_embeddings.pt          # full per-residue ESM-2 embeddings, if available
    gb1_features_k5.pt              # optional processed feature cache, preferred for quick runs
    gb1_splits.csv                  # optional metadata CSV if not stored inside the .pt file
  notebooks/
    GB1_ESM2_quickstart_best_CNN.ipynb
```

The notebook first tries to load `data/gb1_features_k5.pt`. If that does not exist, it rebuilds the features from `data/gb1_esm2_embeddings.pt`.


In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
# For GitHub/Colab quickstart, upload or clone the repo and keep data under ./data.
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"

# If running from Google Drive instead, uncomment and edit these lines:
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_ROOT = Path('/content/drive/MyDrive/CBMF 4761/GB1_ESM2_repo')
# DATA_DIR = REPO_ROOT / 'data'

EMB_PATH = DATA_DIR / "gb1_esm2_embeddings.pt"
PROCESSED_FEATURES_PATH = DATA_DIR / "gb1_features_k5.pt"
METADATA_CSV_PATH = DATA_DIR / "gb1_splits.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Data directory:", DATA_DIR.resolve())
print("Processed features exists:", PROCESSED_FEATURES_PATH.exists())
print("Full embeddings exists:", EMB_PATH.exists())
print("Metadata CSV exists:", METADATA_CSV_PATH.exists())

# ---------------------------------------------------------------------
# Project-specific constants
# ---------------------------------------------------------------------
MUT_POSITIONS = [38, 39, 40, 53]  # zero-based residue positions used in the project notebooks
WINDOW_RADIUS = 5                 # local ESM-2 feature window around mutation positions
SPLIT_COLS = ["one_vs_rest", "two_vs_rest", "three_vs_rest"]
TARGET_COL = "fitness"


## 3. Utility functions for reproducibility, loading, and metrics

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def load_torch_obj(path):
    """Load a saved PyTorch object on CPU."""
    return torch.load(path, map_location="cpu", weights_only=False)


def get_first_existing(obj, keys):
    """Return the first available key from a dict-like object."""
    for key in keys:
        if key in obj:
            return obj[key]
    raise KeyError(f"None of these keys were found: {keys}. Available keys: {list(obj.keys())}")


def build_df_from_embedding_obj(obj):
    """Build the GB1 metadata dataframe from the saved embedding object."""
    return pd.DataFrame({
        "Variants": get_first_existing(obj, ["Variants", "variant", "variants"]),
        "sequence": get_first_existing(obj, ["sequence", "Sequence", "sequences"]),
        "fitness": get_first_existing(obj, ["fitness", "Fitness"]),
        "one_vs_rest": get_first_existing(obj, ["one_vs_rest"]),
        "two_vs_rest": get_first_existing(obj, ["two_vs_rest"]),
        "three_vs_rest": get_first_existing(obj, ["three_vs_rest"]),
    })


def compute_spearman(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    if len(np.unique(y_true)) < 2 or len(np.unique(y_pred)) < 2:
        return np.nan
    return float(spearmanr(y_true, y_pred).correlation)


def compute_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return {
        "mse": float(np.mean((y_true - y_pred) ** 2)),
        "mae": float(np.mean(np.abs(y_true - y_pred))),
        "spearman": compute_spearman(y_true, y_pred),
    }


## 4. Build or load the final fused-model features

The preferred quickstart path is to load `gb1_features_k5.pt`, which should contain:

- `df_full`: metadata with fitness and split columns
- `X_seq_mean`: full-sequence mean ESM-2 embedding, shape `(n_samples, 640)`
- `X_window_k5`: local mutation-window tensor, shape `(n_samples, n_window_positions, 640)`
- `feature_info`: metadata describing the feature construction

If that cache is not present, the notebook reconstructs those tensors from `gb1_esm2_embeddings.pt`.


In [ ]:
def get_window_positions(mut_positions, seq_len, radius=5):
    """Return sorted unique residue positions within +/- radius of each mutation position."""
    pos_set = set(mut_positions)
    for pos in mut_positions:
        for offset in range(1, radius + 1):
            if pos - offset >= 0:
                pos_set.add(pos - offset)
            if pos + offset < seq_len:
                pos_set.add(pos + offset)
    return sorted(pos_set)


def build_final_features_from_residue_embeddings(X_full_residue, mut_positions, window_radius=5):
    """Build final fused-model features from full per-residue ESM-2 embeddings."""
    X_np = to_numpy(X_full_residue)
    n_samples, seq_len, emb_dim = X_np.shape

    X_seq_mean = X_np.mean(axis=1)
    window_positions = get_window_positions(mut_positions, seq_len=seq_len, radius=window_radius)
    X_window = X_np[:, window_positions, :]

    feature_info = {
        "mut_positions_zero_based": list(mut_positions),
        "window_radius": window_radius,
        "window_positions_zero_based": window_positions,
        "n_window_positions": len(window_positions),
        "seq_len": seq_len,
        "emb_dim": emb_dim,
        "X_seq_mean_shape": tuple(X_seq_mean.shape),
        "X_window_shape": tuple(X_window.shape),
    }
    return X_seq_mean, X_window, feature_info


def load_or_build_features():
    if PROCESSED_FEATURES_PATH.exists():
        print(f"Loading processed features from {PROCESSED_FEATURES_PATH}")
        obj = load_torch_obj(PROCESSED_FEATURES_PATH)

        # Support a few reasonable key names so the cache format can be simple.
        df_full = obj.get("df_full", obj.get("df", None))
        if df_full is None:
            raise KeyError("Processed feature file must contain 'df_full' or 'df'.")
        if not isinstance(df_full, pd.DataFrame):
            df_full = pd.DataFrame(df_full)

        X_seq_mean = obj.get("X_seq_mean", obj.get("X_mean", None))
        X_window = obj.get("X_window_k5", obj.get("X_window", None))
        if X_seq_mean is None or X_window is None:
            raise KeyError("Processed feature file must contain X_seq_mean/X_mean and X_window_k5/X_window.")

        feature_info = obj.get("feature_info", {})
        return df_full, torch.tensor(to_numpy(X_seq_mean), dtype=torch.float32), torch.tensor(to_numpy(X_window), dtype=torch.float32), feature_info

    if not EMB_PATH.exists():
        raise FileNotFoundError(
            "Could not find processed features or full embeddings. Expected one of:
"
            f"  {PROCESSED_FEATURES_PATH}
"
            f"  {EMB_PATH}
"
            "Upload the data files into ./data or edit DATA_DIR above."
        )

    print(f"Loading full ESM-2 embeddings from {EMB_PATH}")
    emb_obj = load_torch_obj(EMB_PATH)

    # Metadata can either be stored inside the embedding object or separately as a CSV.
    if METADATA_CSV_PATH.exists():
        df_full = pd.read_csv(METADATA_CSV_PATH)
    else:
        df_full = build_df_from_embedding_obj(emb_obj)

    X_full_residue = get_first_existing(emb_obj, ["X_residue", "X", "embeddings", "representations"])
    X_seq_mean, X_window, feature_info = build_final_features_from_residue_embeddings(
        X_full_residue=X_full_residue,
        mut_positions=MUT_POSITIONS,
        window_radius=WINDOW_RADIUS,
    )

    return df_full, torch.tensor(X_seq_mean, dtype=torch.float32), torch.tensor(X_window, dtype=torch.float32), feature_info


set_seed(42)
df_full, X_seq_mean, X_window_k5, feature_info = load_or_build_features()

print("Metadata shape:", df_full.shape)
print("X_seq_mean shape:", tuple(X_seq_mean.shape))
print("X_window_k5 shape:", tuple(X_window_k5.shape))
print("Feature info:")
for k, v in feature_info.items():
    print(f"  {k}: {v}")

print("
Split counts:")
for split in SPLIT_COLS:
    print(split)
    print(df_full[split].value_counts(dropna=False))


## 5. Create train/test tensors and DataLoaders

In [ ]:
class HybridRegressionDataset(Dataset):
    """Two-input dataset for fused global/local models."""
    def __init__(self, X_window, X_mean, y):
        self.X_window = X_window
        self.X_mean = X_mean
        self.y = y

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        return {
            "window": self.X_window[idx],
            "mean": self.X_mean[idx],
            "y": self.y[idx],
        }


def make_hybrid_split_tensors(X_window, X_mean, df, split_cols=SPLIT_COLS, target_col=TARGET_COL):
    y = torch.tensor(df[target_col].values, dtype=torch.float32)
    split_data = {}

    for split_name in split_cols:
        labels = df[split_name].astype(str).str.lower().str.strip()
        train_mask = torch.tensor((labels == "train").values, dtype=torch.bool)
        test_mask = torch.tensor((labels == "test").values, dtype=torch.bool)

        split_data[split_name] = {
            "X_window_train": X_window[train_mask],
            "X_mean_train": X_mean[train_mask],
            "y_train": y[train_mask],
            "X_window_test": X_window[test_mask],
            "X_mean_test": X_mean[test_mask],
            "y_test": y[test_mask],
            "train_mask": train_mask,
            "test_mask": test_mask,
        }
    return split_data


def make_hybrid_loaders(split_tensors_hybrid, split_name, batch_size=128, test_batch_size=512):
    d = split_tensors_hybrid[split_name]
    train_ds = HybridRegressionDataset(d["X_window_train"], d["X_mean_train"], d["y_train"])
    test_ds = HybridRegressionDataset(d["X_window_test"], d["X_mean_test"], d["y_test"])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=test_batch_size, shuffle=False)
    return train_loader, test_loader


split_tensors_hybrid = make_hybrid_split_tensors(X_window_k5, X_seq_mean, df_full)

for split_name in SPLIT_COLS:
    d = split_tensors_hybrid[split_name]
    print(split_name)
    print("  X_window train/test:", tuple(d["X_window_train"].shape), tuple(d["X_window_test"].shape))
    print("  X_mean train/test  :", tuple(d["X_mean_train"].shape), tuple(d["X_mean_test"].shape))
    print("  y train/test       :", tuple(d["y_train"].shape), tuple(d["y_test"].shape))


## 6. Define the fused CNN model

In [ ]:
class HybridGB1CNNLocalModel(nn.Module):
    """
    Fused GB1 model with:
      - global MLP branch over the full-sequence mean embedding
      - local 1D CNN branch over the mutation-window residue embeddings
      - fusion MLP head for regression
    """

    def __init__(
        self,
        embed_dim: int,
        local_len: int,
        global_hidden_dims=(128, 64),
        cnn_channels=384,
        cnn_kernel_size=7,
        cnn_num_layers=1,
        fusion_hidden_dims=(64,),
        dropout: float = 0.1,
        activation: str = "gelu",
        local_pool: str = "max",
    ):
        super().__init__()

        if cnn_kernel_size % 2 == 0:
            raise ValueError("Use an odd cnn_kernel_size so padding preserves local length.")
        if local_pool not in {"max", "mean"}:
            raise ValueError("local_pool must be 'max' or 'mean'.")

        self.embed_dim = embed_dim
        self.local_len = local_len
        self.local_pool = local_pool

        self.global_branch = self._make_mlp(embed_dim, global_hidden_dims, dropout, activation)
        self.local_cnn = self._make_cnn(embed_dim, cnn_channels, cnn_kernel_size, cnn_num_layers, dropout, activation)
        self.fusion_head = self._make_mlp(global_hidden_dims[-1] + cnn_channels, fusion_hidden_dims, dropout, activation)
        self.output_layer = nn.Linear(fusion_hidden_dims[-1], 1)

    def _get_activation(self, activation):
        activation = activation.lower()
        if activation == "relu":
            return nn.ReLU()
        if activation == "gelu":
            return nn.GELU()
        if activation == "silu":
            return nn.SiLU()
        raise ValueError(f"Unknown activation: {activation}")

    def _make_mlp(self, input_dim, hidden_dims, dropout, activation):
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(self._get_activation(activation))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        return nn.Sequential(*layers)

    def _make_cnn(self, input_channels, hidden_channels, kernel_size, num_layers, dropout, activation):
        layers = []
        padding = kernel_size // 2
        in_channels = input_channels
        for _ in range(num_layers):
            layers.append(nn.Conv1d(in_channels, hidden_channels, kernel_size=kernel_size, padding=padding))
            layers.append(self._get_activation(activation))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_channels = hidden_channels
        return nn.Sequential(*layers)

    def forward(self, x_mean, x_window):
        if x_mean.ndim != 2:
            raise ValueError(f"x_mean should be (batch, embed_dim), got {tuple(x_mean.shape)}")
        if x_window.ndim != 3:
            raise ValueError(f"x_window should be (batch, local_len, embed_dim), got {tuple(x_window.shape)}")

        global_repr = self.global_branch(x_mean)

        # Conv1d expects (batch, channels, sequence_length), so embedding dim is channels.
        local_token_repr = self.local_cnn(x_window.transpose(1, 2))
        if self.local_pool == "max":
            local_repr = local_token_repr.max(dim=-1).values
        else:
            local_repr = local_token_repr.mean(dim=-1)

        combined = torch.cat([global_repr, local_repr], dim=-1)
        hidden = self.fusion_head(combined)
        return self.output_layer(hidden).squeeze(-1)


## 7. Train and evaluate one split

In [ ]:
BEST_CNN_CONFIG = {
    "seed": 42,
    "global_hidden_dims": (128, 64),
    "cnn_channels": 384,
    "cnn_kernel_size": 7,
    "cnn_num_layers": 1,
    "fusion_hidden_dims": (64,),
    "dropout": 0.1,
    "activation": "gelu",
    "local_pool": "max",
    "lr": 1e-3,
    "weight_decay": 1e-3,
    "batch_size": 128,
    "test_batch_size": 512,
    "max_epochs": 150,
    "patience": 25,
}


def evaluate_hybrid_model(model, loader, device=DEVICE):
    model.eval()
    preds, ys = [], []
    loss_fn = nn.MSELoss(reduction="sum")
    total_loss = 0.0
    n = 0

    with torch.no_grad():
        for batch in loader:
            x_window = batch["window"].to(device)
            x_mean = batch["mean"].to(device)
            y = batch["y"].to(device)

            pred = model(x_mean=x_mean, x_window=x_window)
            total_loss += loss_fn(pred, y).item()
            n += y.numel()
            preds.append(pred.detach().cpu())
            ys.append(y.detach().cpu())

    y_true = torch.cat(ys).numpy()
    y_pred = torch.cat(preds).numpy()
    metrics = compute_regression_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / max(n, 1)
    return metrics, y_true, y_pred


def train_one_hybrid_cnn_split(split_name, config=BEST_CNN_CONFIG, verbose=True):
    set_seed(config.get("seed", 42))

    train_loader, test_loader = make_hybrid_loaders(
        split_tensors_hybrid,
        split_name=split_name,
        batch_size=config["batch_size"],
        test_batch_size=config.get("test_batch_size", 512),
    )

    embed_dim = split_tensors_hybrid[split_name]["X_mean_train"].shape[-1]
    local_len = split_tensors_hybrid[split_name]["X_window_train"].shape[1]

    model = HybridGB1CNNLocalModel(
        embed_dim=embed_dim,
        local_len=local_len,
        global_hidden_dims=config["global_hidden_dims"],
        cnn_channels=config["cnn_channels"],
        cnn_kernel_size=config["cnn_kernel_size"],
        cnn_num_layers=config["cnn_num_layers"],
        fusion_hidden_dims=config["fusion_hidden_dims"],
        dropout=config["dropout"],
        activation=config.get("activation", "gelu"),
        local_pool=config.get("local_pool", "max"),
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    loss_fn = nn.MSELoss()

    history = []
    best_spearman = -np.inf
    best_epoch = None
    best_state_dict = None
    best_summary = None
    epochs_without_improvement = 0

    for epoch in range(1, config.get("max_epochs", 150) + 1):
        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for batch in train_loader:
            x_mean = batch["mean"].to(DEVICE)
            x_window = batch["window"].to(DEVICE)
            y = batch["y"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred = model(x_mean=x_mean, x_window=x_window)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * y.numel()
            train_n += y.numel()

        train_eval, _, _ = evaluate_hybrid_model(model, train_loader, device=DEVICE)
        test_eval, _, _ = evaluate_hybrid_model(model, test_loader, device=DEVICE)

        row = {
            "epoch": epoch,
            "train_loss": train_loss_sum / max(train_n, 1),
            "train_eval_loss": train_eval["loss"],
            "test_loss": test_eval["loss"],
            "train_spearman": train_eval["spearman"],
            "test_spearman": test_eval["spearman"],
            "train_mae": train_eval["mae"],
            "test_mae": test_eval["mae"],
        }
        history.append(row)

        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(
                f"{split_name} | epoch {epoch:03d} | "
                f"train loss {row['train_eval_loss']:.4f} | "
                f"test loss {row['test_loss']:.4f} | "
                f"test Spearman {row['test_spearman']:.4f}"
            )

        # This mirrors the exploratory notebooks by tracking best test Spearman.
        # For a formal ML workflow, this should be replaced with validation-based model selection.
        if test_eval["spearman"] > best_spearman:
            best_spearman = test_eval["spearman"]
            best_epoch = epoch
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_summary = {
                "model": "Fused CNN",
                "split": split_name,
                "n_train": int(split_tensors_hybrid[split_name]["y_train"].numel()),
                "n_test": int(split_tensors_hybrid[split_name]["y_test"].numel()),
                "best_epoch": int(best_epoch),
                "best_spearman": float(test_eval["spearman"]),
                "best_train_loss": float(train_eval["loss"]),
                "best_test_loss": float(test_eval["loss"]),
                "best_train_mae": float(train_eval["mae"]),
                "best_test_mae": float(test_eval["mae"]),
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= config.get("patience", 25):
            if verbose:
                print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
            break

    history_df = pd.DataFrame(history)
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    _, y_test_np, pred_test_np = evaluate_hybrid_model(model, test_loader, device=DEVICE)

    return {
        "summary": best_summary,
        "history": history_df,
        "model": model,
        "config": dict(config),
        "y_test": y_test_np,
        "pred_test": pred_test_np,
    }


## 8. Run the best fused CNN across all three splits

This cell is the main reproduction run for the GitHub quickstart notebook.


In [ ]:
results = []
histories = {}
predictions = {}

for split_name in SPLIT_COLS:
    print("=" * 100)
    print(f"Training best fused CNN on {split_name}")
    print("=" * 100)

    result = train_one_hybrid_cnn_split(split_name, config=BEST_CNN_CONFIG, verbose=True)
    results.append(result["summary"])
    histories[split_name] = result["history"]
    predictions[split_name] = {
        "y_test": result["y_test"],
        "pred_test": result["pred_test"],
    }

results_df = pd.DataFrame(results)
display(results_df)


## 9. Save quickstart outputs

In [ ]:
OUTPUT_DIR = REPO_ROOT / "outputs" / "quickstart_best_cnn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(OUTPUT_DIR / "best_cnn_split_results.csv", index=False)

for split_name, hist in histories.items():
    hist.to_csv(OUTPUT_DIR / f"history_{split_name}.csv", index=False)
    pred_df = pd.DataFrame(predictions[split_name])
    pred_df.to_csv(OUTPUT_DIR / f"predictions_{split_name}.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR.resolve())


## 10. Optional: quick results plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_df = results_df.set_index("split").loc[SPLIT_COLS]
ax.bar(plot_df.index, plot_df["best_spearman"])
ax.set_ylabel("Best test Spearman correlation")
ax.set_xlabel("FLIP split")
ax.set_title("Best fused CNN performance across GB1 splits")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


## Notes

- This notebook intentionally trains only one final CNN configuration instead of rerunning the full grid search.
- The model selection logic mirrors the exploratory project notebooks by reporting the best test Spearman observed during training. For a more formal benchmark, a separate validation split should be used for early stopping/model selection and the test set should be evaluated once.
- If the full ESM-2 embedding file is too large for GitHub, include `gb1_features_k5.pt` instead and document where the full embedding file can be regenerated or downloaded.
